# Enrichissement Géographique - Phase 2

Ce notebook permet d'enrichir les données immobilières avec des informations géographiques.

## Objectifs
- Géocoder les villes pour obtenir latitude/longitude
- Calculer les distances (centre-ville, aéroport, plage)
- Récupérer les Points d'Intérêt (POI) : écoles, mosquées, commerces, hôpitaux
- Exporter vers `data/processed/enriched_data.csv`

In [ ]:
# Section A - Imports et Configuration
import sys
import os
from pathlib import Path

# Ajouter le répertoire src au path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
import numpy as np
from tqdm import tqdm
import time

# Imports pour l'enrichissement géographique
from src.geo.enrichment import (
    geocode_location,
    calculate_distances,
    get_pois_around,
    enrich_dataframe,
)
from src.geo.config import (
    QUARTIERS_NOUAKCHOTT,
    CENTRE_VILLE_KSAR,
    AEROPORT_OUMTOUNSY,
    PLAGE_NOUAKCHOTT,
)

print("✓ Imports réussis")
print(f"✓ Répertoire de travail: {project_root}")
print(f"✓ Nombre de quartiers dans le mapping: {len(QUARTIERS_NOUAKCHOTT)}")

ModuleNotFoundError: No module named 'src'

## Section B - Chargement des données

In [ ]:
# Charger les données brutes
raw_data_path = project_root / 'data' / 'raw' / 'raw_data.csv'

if not raw_data_path.exists():
    raise FileNotFoundError(f"Fichier non trouvé: {raw_data_path}")

df = pd.read_csv(raw_data_path)
print(f"✓ Données chargées: {len(df)} lignes")
print(f"✓ Colonnes: {list(df.columns)}")
print(f"\nVilles uniques dans le dataset:")
print(df['ville'].value_counts().head(10))
print(f"\nValeurs manquantes dans 'ville': {df['ville'].isna().sum()}")

## Section C - Test de géocodage sur quelques exemples

In [ ]:
# Tester le géocodage sur quelques villes
test_villes = ['Tevragh Zeina', 'Ksar', 'Nouadhibou', 'Arafat']

print("Test de géocodage:\n")
for ville in test_villes:
    coords = geocode_location(ville)
    if coords:
        lat, lon = coords
        print(f"✓ {ville}: ({lat}, {lon})")
        
        # Tester le calcul des distances
        distances = calculate_distances(lat, lon)
        print(f"  - Distance au centre-ville: {distances['dist_centre_ville_km']:.2f} km")
        print(f"  - Distance à l'aéroport: {distances['dist_aeroport_km']:.2f} km")
        print(f"  - Distance à la plage: {distances['dist_plage_km']:.2f} km")
        
        # Tester la récupération des POI (peut être long)
        print(f"  - Récupération des POI...")
        pois = get_pois_around(lat, lon)
        print(f"    Écoles: {pois['nb_ecoles_1km']}, Mosquées: {pois['nb_mosquees_1km']}, "
              f"Commerces: {pois['nb_commerces_1km']}, Hôpitaux: {pois['nb_hopitaux_1km']}")
    else:
        print(f"✗ {ville}: Géocodage échoué")
    print()

## Section D - Enrichissement complet du dataset

**Attention**: Cette étape peut prendre plusieurs heures pour un grand dataset (10k+ lignes) car :
- Nominatim limite à 1 requête/seconde
- Overpass API peut être lent pour récupérer les POI
- Chaque ligne nécessite plusieurs requêtes API

Pour tester rapidement, vous pouvez utiliser un échantillon du dataset.

In [ ]:
# Option 1: Enrichir tout le dataset (peut prendre plusieurs heures)
# df_enriched = enrich_dataframe(df, progress_bar=True)

# Option 2: Tester sur un échantillon (recommandé pour le développement)
SAMPLE_SIZE = 100  # Modifier selon vos besoins
df_sample = df.head(SAMPLE_SIZE).copy()
print(f"Enrichissement d'un échantillon de {len(df_sample)} lignes...")
df_enriched = enrich_dataframe(df_sample, progress_bar=True)

print(f"\n✓ Enrichissement terminé!")
print(f"\nStatistiques:")
print(f"  - Lignes avec coordonnées GPS: {df_enriched['latitude'].notna().sum()}")
print(f"  - Lignes avec distances: {df_enriched['dist_centre_ville_km'].notna().sum()}")
print(f"  - Lignes avec POI: {df_enriched['nb_total_pois_1km'].sum() > 0}")

In [ ]:
# Afficher un aperçu des données enrichies
print("Aperçu des données enrichies:")
print(df_enriched[['ville', 'latitude', 'longitude', 'dist_centre_ville_km', 
                   'nb_ecoles_1km', 'nb_mosquees_1km', 'nb_commerces_1km', 
                   'nb_hopitaux_1km', 'nb_total_pois_1km']].head(10))

## Section E - Visualisation cartographique

In [ ]:
# Visualisation avec Folium (optionnel - nécessite folium)
try:
    import folium
    from folium import plugins
    
    # Créer une carte centrée sur Nouakchott
    m = folium.Map(location=[18.0866, -15.9750], zoom_start=12)
    
    # Ajouter les points de référence
    folium.Marker(
        CENTRE_VILLE_KSAR,
        popup='Centre-ville (Ksar)',
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(m)
    
    folium.Marker(
        AEROPORT_OUMTOUNSY,
        popup='Aéroport Oumtounsy',
        icon=folium.Icon(color='blue', icon='plane')
    ).add_to(m)
    
    folium.Marker(
        PLAGE_NOUAKCHOTT,
        popup='Plage de Nouakchott',
        icon=folium.Icon(color='green', icon='tint')
    ).add_to(m)
    
    # Ajouter les annonces avec coordonnées
    df_with_coords = df_enriched[df_enriched['latitude'].notna()]
    
    for idx, row in df_with_coords.iterrows():
        popup_text = f"""
        <b>{row.get('titre', 'N/A')[:50]}</b><br>
        Ville: {row.get('ville', 'N/A')}<br>
        Prix: {row.get('prix', 'N/A')} MRU<br>
        Surface: {row.get('surface_m2', 'N/A')} m²<br>
        Distance centre: {row.get('dist_centre_ville_km', 'N/A'):.2f} km<br>
        POI: {row.get('nb_total_pois_1km', 0)}
        """
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=5,
            popup=folium.Popup(popup_text, max_width=200),
            color='blue',
            fill=True,
            fillColor='blue',
            fillOpacity=0.6
        ).add_to(m)
    
    # Afficher la carte
    print(f"✓ Carte créée avec {len(df_with_coords)} points")
    m
    
except ImportError:
    print("⚠ Folium n'est pas installé. Installation: pip install folium")
    print("  La visualisation cartographique est optionnelle.")
except Exception as e:
    print(f"⚠ Erreur lors de la création de la carte: {e}")

In [ ]:
# Créer le répertoire de destination si nécessaire
output_dir = project_root / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

# Sauvegarder les données enrichies
output_path = output_dir / 'enriched_data.csv'
df_enriched.to_csv(output_path, index=False, encoding='utf-8')

print(f"✓ Données enrichies sauvegardées dans: {output_path}")
print(f"✓ Nombre de lignes: {len(df_enriched)}")
print(f"✓ Nombre de colonnes: {len(df_enriched.columns)}")
print(f"\nNouvelles colonnes ajoutées:")
new_columns = ['latitude', 'longitude', 'dist_centre_ville_km', 'dist_aeroport_km', 
               'dist_plage_km', 'nb_ecoles_1km', 'nb_mosquees_1km', 
               'nb_commerces_1km', 'nb_hopitaux_1km', 'nb_total_pois_1km']
for col in new_columns:
    if col in df_enriched.columns:
        non_null = df_enriched[col].notna().sum() if df_enriched[col].dtype != 'int64' else (df_enriched[col] > 0).sum()
        print(f"  - {col}: {non_null} valeurs non nulles")

## Notes importantes

1. **Performance**: Pour enrichir tout le dataset (10k+ lignes), prévoir plusieurs heures d'exécution
2. **Rate limiting**: Nominatim limite à 1 requête/seconde - respecté automatiquement
3. **Cache**: Les résultats de géocodage sont mis en cache pour éviter les requêtes redondantes
4. **POI**: La récupération des POI peut être lente - chaque ligne nécessite 4 requêtes Overpass API
5. **Valeurs manquantes**: Les villes non trouvées auront des valeurs NaN pour les coordonnées et distances

## Prochaines étapes

- Phase 3: EDA complète sur les données enrichies
- Phase 4: Feature Engineering & Modélisation